# ДЗ 4. Операции, axis и broadcasting на данных «Дала Маркет»
Срок: **до понедельника 16.11, 19:00**.
Сдача: папка `hw04` репозитория `dsa-homework` (`hw04.ipynb`): `git add hw04` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем. Небольшой цикл по городам или категориям при сборке таблицы допустим.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())
```

In [1]:
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)

[1. 2. 3.] [0 1 1]


Ваши ответы и объяснения:

1. 
2. 
3. 
4. 
5. 

In [2]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [3]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

(5463,) ['2026-09' '2026-10']


In [11]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
print(first)
keep = np.zeros(len(oid), dtype=bool)
print(oid)
print(np.zeros(4))
print(keep)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

[1989 1428 2514 ... 1001 2268 2966]
['101674' '101214' '102424' ... '104615' '103460' '103501']
[0. 0. 0. 0.]
[False False False ... False False False]
строк всего: 5463 | в чистых данных: 4739


## [База] 1. Выручка по чистым данным
Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?

In [14]:
clean.sum()


np.int64(4739)

In [19]:
amount[month== '2026-09'].sum()


np.float64(60704470.0)

In [20]:
amount[month== '2026-10'].sum()

np.float64(49178890.0)

In [21]:
amount[month== '2026-10'].sum() +amount[month== '2026-09'].sum()

np.float64(109883360.0)

In [22]:
amount.sum()

np.float64(109883360.0)

In [25]:
round(amount[month== '2026-10'].sum() *100/amount[month== '2026-09'].sum()-100,2)

np.float64(-18.99)

## [База] 2. Таблица «категория × месяц»
Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?

In [42]:
categories =np.unique(cat)
categories 

array(['Дом', 'Книги', 'Красота', 'Одежда', 'Электроника'], dtype='<U11')

In [65]:
a = [1, 2, 3]
a


[1, 2, 3]

In [67]:
b = ['a','b','c']
b

['a', 'b', 'c']

In [69]:
arr = [a,b]
arr

[[1, 2, 3], ['a', 'b', 'c']]

In [75]:
arr_np = np.array(arr)
arr_np

array([['1', '2', '3'],
       ['a', 'b', 'c']], dtype='<U21')

In [76]:
arr_np.shape

(2, 3)

In [47]:
months = np.unique(month)
months

array(['2026-09', '2026-10'], dtype='<U7')

In [48]:
month

array(['2026-09', '2026-09', '2026-09', ..., '2026-10', '2026-10',
       '2026-10'], shape=(5463,), dtype='<U7')

In [96]:
table = np.array([[amount[clean&(m == month)&(c == cat)].sum() for m in months] for c in categories])
for c,t in zip(categories, table):
    print(f"{c:>12}  {t[0]:<15}  {t[1]:<15}")

         Дом  4997730.0        5125790.0      
       Книги  1065300.0        1140910.0      
     Красота  2610950.0        2678990.0      
      Одежда  9167100.0        8505730.0      
 Электроника  17402330.0       16968100.0     


In [104]:
table.sum(axis=1).argmax()

np.int64(4)

In [105]:
categories[table.sum(axis=1).argmax()]

np.str_('Электроника')

## [База] 3. Средний чек по категориям в октябре
Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?

In [114]:
table2 = np.array([[(clean&(m == month)&(c == cat)).sum() for m in months] for c in categories])
table2

array([[476, 476],
       [265, 299],
       [422, 435],
       [678, 652],
       [531, 505]])

In [121]:
#Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?
mean_table = table/table2
mean_table

array([[10499.43277311, 10768.46638655],
       [ 4020.        ,  3815.75250836],
       [ 6187.08530806,  6158.59770115],
       [13520.79646018, 13045.59815951],
       [32772.74952919, 33600.1980198 ]])

In [124]:
categories[mean_table.sum(axis = 1).argmax()]

np.str_('Электроника')

## [Стандарт] 4. Доля городов в выручке месяца
Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?

In [129]:
table_cities = np.array([[amount[clean & (city == c) & (month == m)].sum() for m in months]
                  for c in cities])
amount_part = table_cities / table_cities.sum(axis=0)
amount_part

array([[0.07311892, 0.07380754],
       [0.41145479, 0.42542778],
       [0.25633785, 0.25713694],
       [0.02621738, 0.02817762],
       [0.10352347, 0.09795837],
       [0.12934759, 0.11749176]])

In [130]:
amount_part.sum(axis = 0)

array([1., 1.])

In [132]:
for ct, t in zip(cities, table_cities):
    print(f"{ct:>12} {t[0]}  {t[1]}")

      Актобе 2576960.0  2540420.0
      Алматы 14501070.0  14643020.0
      Астана 9034220.0  8850530.0
      Атырау 923990.0  969860.0
   Караганда 3648520.0  3371680.0
     Шымкент 4558650.0  4044010.0


In [142]:
# Как изменилась доля Шымкента?
amount_part[[c=="Шымкент" for c in cities]][0][1]*100/amount_part[[c=="Шымкент" for c in cities]][0][0] -100

np.float64(-9.165865189264167)

In [145]:
table_cities[:, 1]

array([ 2540420., 14643020.,  8850530.,   969860.,  3371680.,  4044010.])

## [Стандарт] 5. Кто дал падение
Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
Напишите менеджеру одну фразу: где искать причину.

In [144]:
city_change = table_cities[:, 1] - table_cities[:, 0]
city_change

array([ -36540.,  141950., -183690.,   45870., -276840., -514640.])

In [146]:
city_change.sum()

np.float64(-823890.0)

In [149]:
city_change/city_change.sum()

array([ 0.04435058, -0.17229242,  0.22295452, -0.05567491,  0.33601573,
        0.62464649])

In [152]:
contribution = city_change / city_change.sum()
shymkent_pct = contribution[cities == 'Шымкент'][0] * 100
shymkent_pct

np.float64(62.46464940708104)

## [Стандарт] 6. Исправьте код от AI
AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
```python
share = table / table.sum(axis=1)
```

In [153]:
share = table_cities / table_cities.sum(axis=1)

ValueError: operands could not be broadcast together with shapes (6,2) (6,) 

In [154]:
table_cities


array([[ 2576960.,  2540420.],
       [14501070., 14643020.],
       [ 9034220.,  8850530.],
       [  923990.,   969860.],
       [ 3648520.,  3371680.],
       [ 4558650.,  4044010.]])

Код падает, потому что размерность матриц не подходит

In [155]:
table_cities.sum(axis=1)

array([ 5117380., 29144090., 17884750.,  1893850.,  7020200.,  8602660.])

In [156]:
table_cities.sum(axis=1).shape

(6,)

In [158]:
share = table_cities / table_cities.sum(axis=0)
share

array([[0.07311892, 0.07380754],
       [0.41145479, 0.42542778],
       [0.25633785, 0.25713694],
       [0.02621738, 0.02817762],
       [0.10352347, 0.09795837],
       [0.12934759, 0.11749176]])

## [Челлендж] 7. Стандартизация суммы заказа
Для чистых оплаченных заказов посчитайте z-оценку: `z = (x - x.mean()) / x.std()`.
Сколько заказов с |z| > 3 и какая это доля? Проверьте, что у `z` среднее ≈ 0 и std = 1.

In [ ]:
# 7

## [Челлендж] 8. Масштабирование признаков (заготовка для ML)
Соберите матрицу `X` из двух столбцов: `amount` и `days` для чистых заказов без пропусков в `days` (`np.column_stack`).
Приведите каждый столбец к диапазону [0, 1] одной строкой: `(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))`.
Проверьте min и max каждого столбца. Объясните своими словами, зачем модели нужны признаки в одном масштабе.

In [ ]:
# 8